# 🚲 Bicycle RTI Hackathon — One-Click Deployment

This notebook deploys the **complete** Bicycle Real-Time Intelligence solution into your workspace.

## What Gets Deployed (23 items)
| Stage | Items |
|-------|-------|
| 1. Storage | 4 Lakehouses |
| 2. Real-Time | 1 Eventhouse → 1 KQL Database (depends on Eventhouse) |
| 3. Compute & Ingestion | 9 Notebooks, 2 Eventstreams |
| 4. Analytics | 2 Semantic Models, 1 Pipeline |
| 5. Presentation | 1 Report, 1 KQL Dashboard, 2 Data Agents, 2 Activators |

## Before you start (one-time setup)

1. Create a **temporary Lakehouse** in your workspace (e.g. name it `deploy_staging`)
2. Open the lakehouse → click **Upload → Upload folder**
3. Upload the **`workspace`** folder from the cloned repo
4. You should now see `Files/workspace/` in the lakehouse with ~26 sub-folders
5. **Attach** this lakehouse to this notebook (left sidebar → "Add lakehouse" → select `deploy_staging`)

## Instructions

1. Run **Cell 1** — installs `fabric-cicd` (one-time)
2. Run **Cell 2** — deploys all 23 items from the lakehouse files (no GitHub, no PAT, no auth prompts)
3. Run **Cell 3** — auto-fixes KQL Dashboard URI + Pipeline placeholders
4. Run **Cell 4** — validates deployment
5. See **Cell 5** — remaining manual steps
5. **(Optional)** Delete the `deploy_staging` lakehouse after deployment

In [ ]:
# =============================================================
# CELL 1 — Install dependencies
# =============================================================
# Pip dependency warnings from nni/mlflow/datasets are harmless —
# they are pre-installed Fabric runtime packages, not ours.

%pip install fabric-cicd --quiet

# Restart Python so the newly installed package is importable
try:
    import notebookutils
    notebookutils.session.restartPython()
except NameError:
    print("⚠️  notebookutils not available — restart the kernel manually before Cell 2.")

In [ ]:
# =============================================================
# CELL 2 — Deploy all 26 items from Lakehouse
# =============================================================
import os, shutil, tempfile, json, base64, time, requests
import sempy.fabric as fabric
from fabric_cicd import FabricWorkspace, publish_all_items

# ── Locate workspace/ folder in the attached lakehouse ──
LAKEHOUSE_PATH = "/lakehouse/default/Files/workspace"

if not os.path.isdir(LAKEHOUSE_PATH):
    alt_paths = [
        "/lakehouse/default/Files/RTI-Hackathon-Demo/workspace",
        "/lakehouse/default/Files/RTI-Hackathon-Demo-main/workspace",
    ]
    found = None
    for p in alt_paths:
        if os.path.isdir(p):
            found = p
            break
    if found:
        LAKEHOUSE_PATH = found
    else:
        print("❌ Cannot find workspace/ folder in the attached lakehouse.")
        print("   Expected: /lakehouse/default/Files/workspace/")
        print("   Fix: Open your lakehouse → Upload → Upload folder → select the 'workspace' folder")
        raise FileNotFoundError(f"workspace/ not found at {LAKEHOUSE_PATH}")

all_items = [d for d in os.listdir(LAKEHOUSE_PATH)
             if os.path.isdir(os.path.join(LAKEHOUSE_PATH, d))]
print(f"📂 Found {len(all_items)} items in {LAKEHOUSE_PATH}")

# ── Build item-type index ──
item_index = {}
for folder_name in all_items:
    parts = folder_name.rsplit(".", 1)
    if len(parts) == 2:
        item_index.setdefault(parts[1], []).append(folder_name)
print("   Item types found:", {k: len(v) for k, v in item_index.items()})

ws_id = fabric.get_workspace_id()
print(f"\n🚀 Deploying to workspace: {ws_id}")

# ── Helper: get access token ──
def get_token():
    try:
        return notebookutils.credentials.getToken("https://api.fabric.microsoft.com")
    except Exception:
        try:
            return notebookutils.credentials.getToken("pbi")
        except Exception:
            from notebookutils import credentials
            return credentials.getToken("https://api.fabric.microsoft.com")

def get_kusto_token():
    """Get token scoped for Kusto management endpoint."""
    for scope in ["https://kusto.kusto.windows.net", "https://api.fabric.microsoft.com", "pbi"]:
        try:
            return notebookutils.credentials.getToken(scope)
        except Exception:
            continue
    raise RuntimeError("Could not get Kusto token")

# ── Helper: create isolated temp dir for a fabric-cicd stage ──
def make_stage_dir(type_list, ref_types=None):
    """Copy folders for given item types into a temp dir.
    For ref_types, copy only .platform files (for logicalId resolution)."""
    stage_dir = tempfile.mkdtemp(prefix="rti_stage_")
    stage_ws = os.path.join(stage_dir, "workspace")
    os.makedirs(stage_ws)
    folders = []
    for t in type_list:
        folders.extend(item_index.get(t, []))
    for f in folders:
        shutil.copytree(os.path.join(LAKEHOUSE_PATH, f),
                        os.path.join(stage_ws, f))
    # Add reference-only .platform files for logicalId resolution
    if ref_types:
        for t in ref_types:
            for f in item_index.get(t, []):
                if os.path.exists(os.path.join(stage_ws, f)):
                    continue  # Already copied as full folder
                ref_dir = os.path.join(stage_ws, f)
                os.makedirs(ref_dir, exist_ok=True)
                src_platform = os.path.join(LAKEHOUSE_PATH, f, ".platform")
                if os.path.exists(src_platform):
                    shutil.copy2(src_platform, os.path.join(ref_dir, ".platform"))
    return stage_dir, stage_ws, folders

# ── Helper: wait for LRO ──
def wait_lro(resp, headers, label, max_wait=180):
    if resp.status_code in (200, 201):
        return resp.json() if resp.text else True
    if resp.status_code != 202:
        return None
    op_url = resp.headers.get("Location")
    if not op_url:
        time.sleep(15)
        return True
    elapsed = 0
    while elapsed < max_wait:
        retry = int(resp.headers.get("Retry-After", 5))
        time.sleep(retry)
        elapsed += retry
        r = requests.get(op_url, headers=headers)
        if r.status_code == 200:
            status = r.json().get("status", "")
            if status == "Succeeded":
                return r.json()
            if status in ("Failed", "Cancelled"):
                print(f"   ❌ {label}: {status} — {r.json().get('error', {}).get('message', '')}")
                return None
    print(f"   ⚠️ {label}: timed out ({max_wait}s)")
    return None

API = "https://api.fabric.microsoft.com/v1"

stage_status = {}

# ═══════════════════════════════════════════════════════════════
# STAGE 1/5: Lakehouses (fabric-cicd)
# ═══════════════════════════════════════════════════════════════
print(f"\n{'='*60}")
print("🚀 Stage 1/5: Lakehouses")
print(f"{'='*60}")
try:
    stage_dir, stage_ws, folders = make_stage_dir(["Lakehouse"])
    print(f"   📦 {', '.join(f.rsplit('.', 1)[0] for f in folders)}")
    ws = FabricWorkspace(workspace_id=ws_id, repository_directory=stage_ws,
                         item_type_in_scope=["Lakehouse"])
    publish_all_items(ws)
    shutil.rmtree(stage_dir, ignore_errors=True)
    stage_status["1_Lakehouses"] = True
    print("   ✅ Stage 1/5 complete")
except Exception as e:
    stage_status["1_Lakehouses"] = False
    print(f"   ❌ Stage 1/5 FAILED: {e}")
    print("   ⚠️ Continuing with remaining stages...")

try:
    stage_dir, stage_ws, folders = make_stage_dir(["Eventhouse"])
    print(f"   📦 {', '.join(f.rsplit('.', 1)[0] for f in folders)}")
    ws = FabricWorkspace(workspace_id=ws_id, repository_directory=stage_ws,
                         item_type_in_scope=["Eventhouse"])
    publish_all_items(ws)
    shutil.rmtree(stage_dir, ignore_errors=True)
    stage_status["2_Eventhouse"] = True
    print("   ✅ Stage 2/5 complete")
except Exception as e:
    stage_status["2_Eventhouse"] = False
    print(f"   ❌ Stage 2/5 FAILED: {e}")
    print("   ⚠️ Continuing with remaining stages...")

# ═══════════════════════════════════════════════════════════════
try:
    token = get_token()
    headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}

    if not stage_status.get("2_Eventhouse"):
        raise RuntimeError("Skipped — Stage 2 (Eventhouse) did not succeed")

    # Find the deployed Eventhouse
    resp = requests.get(f"{API}/workspaces/{ws_id}/eventhouses", headers=headers)
    resp.raise_for_status()
    eventhouse_id = None
    for eh in resp.json().get("value", []):
      if eh["displayName"] == "bikerentaleventhouse":
          eventhouse_id = eh["id"]
          break
          eventhouse_id = eh["id"]
    if not eventhouse_id:
      raise RuntimeError("❌ Eventhouse 'bikerentaleventhouse' not found — Stage 2 may have failed")
    print(f"   Found Eventhouse: {eventhouse_id}")

    # Check if KQL Database already exists
    resp = requests.get(f"{API}/workspaces/{ws_id}/kqlDatabases", headers=headers)
    resp.raise_for_status()
    kqldb_id = None
    for db in resp.json().get("value", []):
      if db["displayName"] == "bikerentaleventhouse":
          kqldb_id = db["id"]
          break

    if kqldb_id:
      print(f"   ℹ️  KQL Database already exists: {kqldb_id}")
    else:
      # Create KQL Database under the Eventhouse
      print("   Creating KQL Database 'bikerentaleventhouse'...")
      create_body = {
          "displayName": "bikerentaleventhouse",
          "creationPayload": {
              "databaseType": "ReadWrite",
              "parentEventhouseItemId": eventhouse_id,
              "oneLakeCachingPeriod": "P36500D",
              "oneLakeStandardStoragePeriod": "P36500D",
          },
      }
      resp = requests.post(f"{API}/workspaces/{ws_id}/kqlDatabases",
                           headers=headers, json=create_body)
      print(f"   Response: HTTP {resp.status_code}")
      if resp.status_code in (200, 201):
          kqldb_id = resp.json().get("id")
          print(f"   ✅ Created KQL Database: {kqldb_id}")
      elif resp.status_code == 202:
          result = wait_lro(resp, headers, "Create KQLDatabase")
          if result:
              # Re-fetch to get the ID
              resp2 = requests.get(f"{API}/workspaces/{ws_id}/kqlDatabases", headers=headers)
              for db in resp2.json().get("value", []):
                  if db["displayName"] == "bikerentaleventhouse":
                      kqldb_id = db["id"]
                      break
              if kqldb_id:
                  print(f"   ✅ Created KQL Database: {kqldb_id}")
              else:
                  raise RuntimeError("❌ KQL Database created but could not find it")
          else:
              raise RuntimeError("❌ Failed to create KQL Database (LRO failed)")
      else:
          error_detail = resp.text[:800]
          print(f"   Error detail: {error_detail}")
          raise RuntimeError(f"❌ Failed to create KQL Database: HTTP {resp.status_code}")

    # Now run the schema KQL commands via the Kusto management API
    print("   Running schema commands...")
    kql_schema_path = None
    for f in item_index.get("KQLDatabase", []):
      schema_file = os.path.join(LAKEHOUSE_PATH, f, "DatabaseSchema.kql")
      if os.path.exists(schema_file):
          kql_schema_path = schema_file
          break

    if kql_schema_path:
      # Wait for query URI to become available (may take a few seconds after creation)
      query_uri = None
      print("   Waiting for KQL Database query URI...")
      for attempt in range(12):  # Retry up to ~60 seconds
          resp = requests.get(f"{API}/workspaces/{ws_id}/kqlDatabases/{kqldb_id}", headers=headers)
          if resp.status_code == 200:
              props = resp.json().get("properties", {})
              query_uri = props.get("queryUri") or props.get("parentEventhouseUri")
              if query_uri:
                  break
          if attempt < 11:
              time.sleep(5)

      if query_uri:
          print(f"   Query URI: {query_uri}")
          # Read schema commands
          with open(kql_schema_path, "r", encoding="utf-8") as sf:
              schema_text = sf.read()

          # Parse individual commands (split on lines starting with .)
          commands = []
          current = []
          for line in schema_text.split("\n"):
              stripped = line.strip()
              if stripped.startswith(".") and current:
                  commands.append("\n".join(current))
                  current = [line]
              elif stripped and not stripped.startswith("//"):
                  current.append(line)
          if current:
              commands.append("\n".join(current))

          # Execute each management command with Kusto-scoped token
          kusto_headers = {
              "Authorization": f"Bearer {get_kusto_token()}",
              "Content-Type": "application/json",
          }
          success_count = 0
          for cmd in commands:
              cmd_clean = cmd.strip()
              if not cmd_clean or cmd_clean.startswith("//"):
                  continue
              try:
                  mgmt_resp = requests.post(
                      f"{query_uri}/v1/rest/mgmt",
                      headers=kusto_headers,
                      json={"csl": cmd_clean, "db": "bikerentaleventhouse"},
                  )
                  if mgmt_resp.status_code == 200:
                      success_count += 1
                  else:
                      first_line = cmd_clean.split("\n")[0][:60]
                      print(f"   ⚠️ Command failed ({mgmt_resp.status_code}): {first_line}...")
                      print(f"      Detail: {mgmt_resp.text[:200]}")
              except Exception as e:
                  print(f"   ⚠️ Command error: {e}")

          print(f"   ✅ Schema: {success_count}/{len(commands)} commands succeeded")
      else:
          print("   ⚠️ Could not get query URI after 60s — schema will need manual setup")
    else:
      print("   ⚠️ DatabaseSchema.kql not found")

    stage_status["3_KQLDatabase"] = True
    print("   ✅ Stage 3/5 complete")
except Exception as e:
    stage_status["3_KQLDatabase"] = False
    print(f"   ❌ Stage 3/5 FAILED: {e}")
    print("   ⚠️ Continuing with remaining stages...")

# ═══════════════════════════════════════════════════════════════
# STAGE 4/5: Semantic Models + Notebooks (fabric-cicd)
# ═══════════════════════════════════════════════════════════════
print(f"\n{'='*60}")
print("🚀 Stage 4/5: Semantic Models + Notebooks")
print(f"{'='*60}")
try:
    stage_dir, stage_ws, _ = make_stage_dir(["SemanticModel", "Notebook"],
                                            ref_types=["KQLDatabase", "Lakehouse", "Eventhouse"])
    deploy_types = ["SemanticModel", "Notebook"]
    folders = []
    for t in deploy_types:
        folders.extend(item_index.get(t, []))
    print(f"   📦 Deploying: {', '.join(f.rsplit('.', 1)[0] for f in folders)}")

    ws = FabricWorkspace(workspace_id=ws_id, repository_directory=stage_ws,
                         item_type_in_scope=deploy_types)
    publish_all_items(ws)
    shutil.rmtree(stage_dir, ignore_errors=True)
    stage_status["4_SM_Notebooks"] = True
    print("   ✅ Stage 4/5 complete")
except Exception as e:
    stage_status["4_SM_Notebooks"] = False
    print(f"   ❌ Stage 4/5 FAILED: {e}")
    print("   ⚠️ Continuing with remaining stages...")

# ═══════════════════════════════════════════════════════════════
# STAGE 5/5: Eventstreams + Pipeline + Report + Dashboard +
#            Agents + Activators (fabric-cicd)
# ═══════════════════════════════════════════════════════════════
print(f"\n{'='*60}")
print("🚀 Stage 5/5: Eventstreams + Analytics + Presentation")
print(f"{'='*60}")
try:
    stage_dir, stage_ws, _ = make_stage_dir(["Eventstream", "DataPipeline", "KQLDashboard", "DataAgent"],
                                            ref_types=["KQLDatabase", "Lakehouse", "Eventhouse"])
    deploy_types = ["Eventstream", "DataPipeline",
                    "KQLDashboard", "DataAgent"]
    folders = []
    for t in deploy_types:
        folders.extend(item_index.get(t, []))
    print(f"   📦 Deploying: {', '.join(f.rsplit('.', 1)[0] for f in folders)}")

    ws = FabricWorkspace(workspace_id=ws_id, repository_directory=stage_ws,
                         item_type_in_scope=deploy_types)
    publish_all_items(ws)
    shutil.rmtree(stage_dir, ignore_errors=True)
    stage_status["5_Remaining"] = True
    print("   ✅ Stage 5/5 complete")
except Exception as e:
    stage_status["5_Remaining"] = False
    print(f"   ❌ Stage 5/5 FAILED: {e}")

# ═══════════════════════════════════════════════════════════════
# DEPLOYMENT SUMMARY
# ═══════════════════════════════════════════════════════════════
print(f"\n{'='*60}")
failed = [k for k, v in stage_status.items() if not v]
if failed:
    print(f"⚠️ DEPLOYMENT PARTIALLY COMPLETE — {len(failed)} stage(s) failed")
else:
    print("✅ DEPLOYMENT COMPLETE — 23 items deployed")
print(f"{'='*60}")
for stage_name, ok in stage_status.items():
    icon = "✅" if ok else "❌"
    print(f"   {icon} {stage_name.replace('_', ': ', 1)}")
if failed:
    print(f"\nTo retry failed stages, re-run this cell — fabric-cicd is idempotent.")
print("\nNext: Run Cell 3 to auto-fix placeholders, then Cell 4 to validate.")

In [ ]:
# =============================================================
# CELL 3 — Auto-Fix Placeholders (KQL Dashboard + Pipeline)
# =============================================================
# After deployment, 2 placeholders remain that fabric-cicd can't resolve:
#   1. __EVENTHOUSE_QUERY_URI__  in the KQL Dashboard
#   2. __SM_REFRESH_CONN__       in the Pipeline (SM refresh activity)
#
# This cell auto-discovers the Eventhouse query URI, patches the
# dashboard, and removes the broken SM refresh activity from the pipeline.

import requests, json, base64, time
import sempy.fabric as fabric

ws_id = fabric.get_workspace_id()

# Get access token (Fabric notebook session)
try:
    token = notebookutils.credentials.getToken("https://api.fabric.microsoft.com")
except Exception:
    try:
        token = notebookutils.credentials.getToken("pbi")
    except Exception:
        from notebookutils import credentials
        token = credentials.getToken("https://api.fabric.microsoft.com")

headers = {
    "Authorization": f"Bearer {token}",
    "Content-Type": "application/json",
}
API = "https://api.fabric.microsoft.com/v1"

print("=" * 60)
print("🔧 AUTO-FIX: Resolving deployment placeholders")
print("=" * 60)

# ── Helper: wait for long-running operation ──
def wait_for_lro(resp, label, max_wait=120):
    """Wait for a 202 long-running operation to complete."""
    if resp.status_code == 200:
        return resp.json()
    if resp.status_code != 202:
        return None
    op_url = resp.headers.get("Location")
    if not op_url:
        time.sleep(10)
        return None
    retry_after = int(resp.headers.get("Retry-After", 5))
    elapsed = 0
    while elapsed < max_wait:
        time.sleep(retry_after)
        elapsed += retry_after
        op_resp = requests.get(op_url, headers=headers)
        if op_resp.status_code == 200:
            result = op_resp.json()
            status = result.get("status", "")
            if status == "Succeeded":
                return result.get("definition", result)
            if status in ("Failed", "Cancelled"):
                print(f"   ⚠️ {label}: {status}")
                return None
    print(f"   ⚠️ {label}: timed out after {max_wait}s")
    return None


# ═══════════════════════════════════════════════════════════════
# STEP 1: Discover Eventhouse query URI
# ═══════════════════════════════════════════════════════════════
print("\n📡 Step 1: Discovering Eventhouse query URI...")

query_uri = None

# Try Eventhouse properties first
resp = requests.get(f"{API}/workspaces/{ws_id}/items?type=Eventhouse", headers=headers)
eventhouse_id = None
if resp.status_code == 200:
    for item in resp.json().get("value", []):
        if item["displayName"] == "bikerentaleventhouse":
            eventhouse_id = item["id"]
            break

if eventhouse_id:
    resp = requests.get(
        f"{API}/workspaces/{ws_id}/eventhouses/{eventhouse_id}", headers=headers
    )
    if resp.status_code == 200:
        props = resp.json().get("properties", {})
        query_uri = props.get("queryServiceUri") or props.get("uri")

# Fallback: try KQL Database properties
if not query_uri:
    resp = requests.get(
        f"{API}/workspaces/{ws_id}/items?type=KQLDatabase", headers=headers
    )
    if resp.status_code == 200:
        for item in resp.json().get("value", []):
            detail = requests.get(
                f"{API}/workspaces/{ws_id}/kqlDatabases/{item['id']}", headers=headers
            )
            if detail.status_code == 200:
                kql_props = detail.json().get("properties", {})
                query_uri = (
                    kql_props.get("queryUri")
                    or kql_props.get("parentEventhouseUri")
                )
                if query_uri:
                    break

if query_uri:
    print(f"   ✅ Eventhouse query URI: {query_uri}")
else:
    print("   ❌ Could not discover query URI — KQL Dashboard will need manual fix")


# ═══════════════════════════════════════════════════════════════
# STEP 2: Patch KQL Dashboard (replace __EVENTHOUSE_QUERY_URI__)
# ═══════════════════════════════════════════════════════════════
if query_uri:
    print("\n📊 Step 2: Patching KQL Dashboard clusterUri...")

    # Find the dashboard
    resp = requests.get(
        f"{API}/workspaces/{ws_id}/items?type=KQLDashboard", headers=headers
    )
    dashboard_id = None
    if resp.status_code == 200:
        for item in resp.json().get("value", []):
            name = item["displayName"]
            if "Fleet Intelligence" in name or "Live Operations" in name:
                dashboard_id = item["id"]
                print(f"   Found dashboard: {name} ({dashboard_id})")
                break

    if dashboard_id:
        # Get current definition
        resp = requests.post(
            f"{API}/workspaces/{ws_id}/items/{dashboard_id}/getDefinition",
            headers=headers,
        )
        definition = wait_for_lro(resp, "Get Dashboard Definition")

        if definition:
            parts = definition.get("definition", definition).get("parts", [])
            patched = False
            new_parts = []
            for part in parts:
                if part.get("path") == "RealTimeDashboard.json":
                    raw = base64.b64decode(part["payload"]).decode("utf-8")
                    if "__EVENTHOUSE_QUERY_URI__" in raw:
                        raw = raw.replace("__EVENTHOUSE_QUERY_URI__", query_uri)
                        patched = True
                    part = dict(part)  # copy
                    part["payload"] = base64.b64encode(
                        raw.encode("utf-8")
                    ).decode("utf-8")
                new_parts.append(part)

            if patched:
                update_body = {"definition": {"parts": new_parts}}
                resp = requests.post(
                    f"{API}/workspaces/{ws_id}/items/{dashboard_id}/updateDefinition",
                    headers=headers,
                    json=update_body,
                )
                result = wait_for_lro(resp, "Update Dashboard")
                if resp.status_code in (200, 201) or result:
                    print("   ✅ KQL Dashboard patched — queries now point to your Eventhouse")
                else:
                    print(f"   ⚠️ Dashboard update returned HTTP {resp.status_code}")
                    print(f"      {resp.text[:300]}")
            else:
                print("   ℹ️  Dashboard already has correct clusterUri (no placeholder)")
        else:
            print("   ⚠️ Could not retrieve dashboard definition")
    else:
        print("   ⚠️ KQL Dashboard not found in workspace")
else:
    print("\n📊 Step 2: SKIPPED (no query URI discovered)")


# ═══════════════════════════════════════════════════════════════
# STEP 3: Patch Pipeline (remove broken SM refresh activity)
# ═══════════════════════════════════════════════════════════════
print("\n🔧 Step 3: Patching Pipeline (removing broken SM refresh)...")

resp = requests.get(
    f"{API}/workspaces/{ws_id}/items?type=DataPipeline", headers=headers
)
pipeline_id = None
if resp.status_code == 200:
    for item in resp.json().get("value", []):
        if item["displayName"] == "PL_BicycleRTI_Medallion":
            pipeline_id = item["id"]
            break

if pipeline_id:
    resp = requests.post(
        f"{API}/workspaces/{ws_id}/items/{pipeline_id}/getDefinition",
        headers=headers,
    )
    definition = wait_for_lro(resp, "Get Pipeline Definition")

    if definition:
        parts = definition.get("definition", definition).get("parts", [])
        patched = False
        new_parts = []
        for part in parts:
            if part.get("path") == "pipeline-content.json":
                raw = base64.b64decode(part["payload"]).decode("utf-8")
                pipeline_def = json.loads(raw)
                activities = pipeline_def.get("properties", {}).get("activities", [])

                original_count = len(activities)
                # Remove SM refresh activity (has __SM_REFRESH_CONN__ placeholder)
                activities = [
                    a for a in activities
                    if a.get("type") != "PBISemanticModelRefresh"
                ]

                if len(activities) < original_count:
                    pipeline_def["properties"]["activities"] = activities
                    removed = original_count - len(activities)
                    raw = json.dumps(pipeline_def, indent=2)
                    patched = True
                    print(f"   Removed {removed} SM refresh activit{'ies' if removed > 1 else 'y'}")

                part = dict(part)
                part["payload"] = base64.b64encode(
                    raw.encode("utf-8")
                ).decode("utf-8")
            new_parts.append(part)

        if patched:
            update_body = {"definition": {"parts": new_parts}}
            resp = requests.post(
                f"{API}/workspaces/{ws_id}/items/{pipeline_id}/updateDefinition",
                headers=headers,
                json=update_body,
            )
            result = wait_for_lro(resp, "Update Pipeline")
            if resp.status_code in (200, 201) or result:
                print("   ✅ Pipeline patched — SM refresh removed (refresh manually after run)")
            else:
                print(f"   ⚠️ Pipeline update returned HTTP {resp.status_code}")
                print(f"      {resp.text[:300]}")
        else:
            print("   ℹ️  Pipeline already clean (no SM refresh activity)")
    else:
        print("   ⚠️ Could not retrieve pipeline definition")
else:
    print("   ⚠️ Pipeline 'PL_BicycleRTI_Medallion' not found")


# ═══════════════════════════════════════════════════════════════
# SUMMARY
# ═══════════════════════════════════════════════════════════════
print(f"\n{'='*60}")
print("✅ AUTO-FIX COMPLETE")
print(f"{'='*60}")
print()
print("What was fixed:")
print("  • KQL Dashboard → clusterUri now points to your Eventhouse")
print("  • Pipeline → SM refresh activity removed (avoids connection error)")
print()
print("Remaining manual steps:")
print("  1. Run the pipeline: PL_BicycleRTI_Medallion (first data load)")
print("  2. Manually refresh Semantic Models after pipeline completes")
print("  3. Run Post_Deploy_Setup.ipynb → creates Ontology + Graph + Agent")
print("     (then the Data Agent graph datasource auto-resolves)")
print("  4. Verify Eventstreams are started")
print("  5. Graph Model → click 'Refresh now'")

In [ ]:
# =============================================================
# CELL 4 — Validate Deployment
# =============================================================
import sempy.fabric as fabric

ws_id = fabric.get_workspace_id()
print(f"Workspace: {ws_id}")

# List all items
items = fabric.list_items()
print(f"\nTotal items: {len(items)}")

# Group by type
from collections import Counter
type_counts = Counter(row['Type'] for _, row in items.iterrows())
print("\nItems by type:")
for t, c in sorted(type_counts.items()):
    print(f"  {t}: {c}")

# Check critical items
expected = [
    "bicycles_gold", "bikerental_bronze_raw", "bicycles_silver", "weather_bronze_raw",
    "bikerentaleventhouse",
    "RTIbikeRental", "RTI-WeatherDemo",
    "PL_BicycleRTI_Medallion",
    "Bicycle RTI Analytics", "Bicycle Ontology Model",
    "Bicycle Fleet Intelligence Agent", "ontology data agent",
]
names = set(items['Display Name'])
print("\nCritical item check:")
for e in expected:
    status = "✅" if e in names else "❌ MISSING"
    print(f"  {status} {e}")

## Remaining Manual Steps

Cell 3 auto-fixed the KQL Dashboard and Pipeline. These steps remain:

### 1. Run the Pipeline (First Data Load)
1. Go to **PL_BicycleRTI_Medallion** pipeline
2. Click **Run** — processes: Silver → Gold → ML → Ontology (~15-25 min)
3. After pipeline completes, **manually refresh** both Semantic Models:
   - Open each model → click **Refresh now**
   - (Cell 3 removed the auto-refresh activity because it needs a connection that can't be created programmatically)

### 2. Ontology + Graph Model (Post-Deploy Notebook)
1. Upload and run `Post_Deploy_Setup.ipynb` to create:
   - Bicycle_Ontology_Model_New (Ontology)
   - Bicycle_Ontology_Model_New_graph (GraphModel)
   - Cycling-Campaign-Agent (OperationsAgent)
2. After ontology is created, the Data Agent graph datasource will be wired automatically

### 3. Refresh Graph Model
1. Open the **Graph Model** → click **Refresh now**
2. This must be done after ontology creation AND after pipeline loads data

### 4. Verify Eventstreams
Both eventstreams should auto-start after deployment:
- **RTIbikeRental** — Bicycle sample data → Lakehouse + Eventhouse
- **RTI-WeatherDemo** — Real-time weather → Eventhouse

If not running, open each Eventstream and click **Start**.


### 5. Activator Rules (Optional)See `docs/ACTIVATOR_SETUP.md` in the repo for rule definitions.

Both Reflex/Activator items are deployed as shells. Add alert rules:

- **BicycleFleet_Activator**: Low stock alert, high demand surge- **Cycling Campaign Activator**: Campaign trigger based on weather